In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\dell\NutriVision")
FOODCOM_DIR = PROJECT_ROOT / "data" / "raw" / "food_com"

print("Folder exists:", FOODCOM_DIR.exists())
print("\nFiles found:")

for file in sorted(FOODCOM_DIR.iterdir()):
    print(f"{file.name:35} {file.stat().st_size / (1024**2):.2f} MB")

Folder exists: True

Files found:
ingr_map.pkl                        0.88 MB
interactions_test.csv               0.50 MB
interactions_train.csv              26.94 MB
interactions_validation.csv         0.28 MB
PP_recipes.csv                      195.42 MB
PP_users.csv                        12.93 MB
RAW_interactions.csv                333.25 MB
RAW_recipes.csv                     280.88 MB


In [2]:
import pandas as pd

RAW_RECIPES_PATH = FOODCOM_DIR / "RAW_recipes.csv"

recipes_df = pd.read_csv(RAW_RECIPES_PATH)

print("Dataset shape:", recipes_df.shape)

print("\nColumns:")
print(recipes_df.columns.tolist())

print("\nFirst 5 recipes:")
display(recipes_df.head())

Dataset shape: (231637, 12)

Columns:
['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']

First 5 recipes:


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [3]:
# --------------------------------------------------
# Step 4.1 - Dataset quality assessment
# --------------------------------------------------

print("Total recipes:", len(recipes_df))

# Missing values
print("\nMissing values:")
display(recipes_df.isna().sum().sort_values(ascending=False))

# Empty or whitespace-only descriptions
description_clean = recipes_df["description"].fillna("").str.strip()

empty_descriptions = description_clean.eq("")

print("\nDescription statistics:")
print("Empty descriptions:", empty_descriptions.sum())
print("Non-empty descriptions:", (~empty_descriptions).sum())
print("Description coverage:", round((~empty_descriptions).mean() * 100, 2), "%")

# Empty ingredients
ingredients_clean = recipes_df["ingredients"].fillna("").str.strip()

empty_ingredients = ingredients_clean.eq("")

print("\nIngredient statistics:")
print("Empty ingredients:", empty_ingredients.sum())
print("Non-empty ingredients:", (~empty_ingredients).sum())

# Duplicate IDs
print("\nDuplicate recipe IDs:", recipes_df["id"].duplicated().sum())

# Description lengths
description_lengths = description_clean.str.split().str.len()

print("\nDescription word count:")
print(description_lengths[~empty_descriptions].describe())

Total recipes: 231637

Missing values:


description       4979
name                 1
minutes              0
id                   0
contributor_id       0
submitted            0
nutrition            0
tags                 0
n_steps              0
steps                0
ingredients          0
n_ingredients        0
dtype: int64


Description statistics:
Empty descriptions: 4982
Non-empty descriptions: 226655
Description coverage: 97.85 %

Ingredient statistics:
Empty ingredients: 0
Non-empty ingredients: 231637

Duplicate recipe IDs: 0

Description word count:
count    226655.000000
mean         36.386186
std          32.225985
min           1.000000
25%          15.000000
50%          28.000000
75%          48.000000
max        1168.000000
Name: description, dtype: float64


In [4]:
# --------------------------------------------------
# Step 5.1 - Inspect recipe descriptions
# --------------------------------------------------

sample_recipes = recipes_df[
    recipes_df["description"].notna()
    & recipes_df["description"].str.strip().ne("")
].sample(10, random_state=42)

for _, row in sample_recipes.iterrows():
    print("=" * 80)
    print("Recipe:", row["name"])
    print("\nDescription:", row["description"])
    print("\nIngredients:", row["ingredients"])
    print()

Recipe: zosia s polish dill pickle soup

Description: this dill pickle soup tastes just like the one they serve at the polish village cafe in hamtramck, mi, a long time polish city next to detroit.  if you ever get a chance, stop into eat there.  if you grew up polish, it feels like you're eating in grandma's basement.  enjoy!

Ingredients: ['chicken broth', 'chicken bouillon cubes', 'carrots', 'potatoes', 'celery', 'dill pickles', 'milk', 'flour', 'egg', 'sour cream', 'salt and pepper', 'fresh parsley', 'fresh dill']

Recipe: blondie bites

Description: i found this recipe while trawling through the food blogs one morning.  a few tweaks later and bite-sized yumminess was emerging from my oven.

Ingredients: ['sweet butter', 'white chocolate', 'brown sugar', 'white sugar', 'eggs', 'flour', 'vanilla extract', 'white chocolate chips', 'macadamia nuts']

Recipe: best ever pizza crust    bread machine

Description: this is something i came up with after modifying several recipes!

Ingredie

In [5]:
# --------------------------------------------------
# Step 6 - Inspect preprocessed Food.com recipes
# --------------------------------------------------

PP_RECIPES_PATH = FOODCOM_DIR / "PP_recipes.csv"

pp_sample = pd.read_csv(PP_RECIPES_PATH, nrows=5)

print("Columns:")
print(pp_sample.columns.tolist())

print("\nShape of sample:")
print(pp_sample.shape)

print("\nFirst 5 rows:")
display(pp_sample)

Columns:
['id', 'i', 'name_tokens', 'ingredient_tokens', 'steps_tokens', 'techniques', 'calorie_level', 'ingredient_ids']

Shape of sample:
(5, 8)

First 5 rows:


,id,i,name_tokens,ingredient_tokens,steps_tokens,techniques,calorie_level,ingredient_ids
0,424415,23,"[40480, 37229, 2911, 1019, 249, 6878, 6878, 28...","[[2911, 1019, 249, 6878], [1353], [6953], [153...","[40480, 40482, 21662, 481, 6878, 500, 246, 161...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[389, 7655, 6270, 1527, 3406]"
1,146223,96900,"[40480, 18376, 7056, 246, 1531, 2032, 40481]","[[17918], [25916], [2507, 6444], [8467, 1179],...","[40480, 40482, 729, 2525, 10906, 485, 43, 8393...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[2683, 4969, 800, 5298, 840, 2499, 6632, 7022,..."
2,312329,120056,"[40480, 21044, 16954, 8294, 556, 10837, 40481]","[[5867, 24176], [1353], [6953], [1301, 11332],...","[40480, 40482, 8240, 481, 24176, 296, 1353, 66...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...",1,"[1257, 7655, 6270, 590, 5024, 1119, 4883, 6696..."
3,74301,168258,"[40480, 10025, 31156, 40481]","[[1270, 1645, 28447], [21601], [27952, 29471, ...","[40480, 40482, 5539, 21601, 1073, 903, 2324, 4...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[7940, 3609, 7060, 6265, 1170, 6654, 5003, 3561]"
4,76272,109030,"[40480, 17841, 252, 782, 2373, 1641, 2373, 252...","[[1430, 11434], [1430, 17027], [1615, 23, 695,...","[40480, 40482, 14046, 1430, 11434, 488, 17027,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[3484, 6324, 7594, 243]"


In [6]:
# --------------------------------------------------
# Step 6.1 - Compare recipe identifiers
# --------------------------------------------------

print("RAW recipe IDs:")
print(recipes_df["id"].head().tolist())

print("\nPP recipe IDs:")
print(pp_sample["id"].tolist() if "id" in pp_sample.columns else "No id column")

RAW recipe IDs:
[137739, 31490, 112140, 59389, 44061]

PP recipe IDs:
[424415, 146223, 312329, 74301, 76272]


In [7]:
# --------------------------------------------------
# Step 6.2 - Compare RAW and PP recipe identifiers
# --------------------------------------------------

pp_df = pd.read_csv(PP_RECIPES_PATH)

raw_ids = set(recipes_df["id"])
pp_ids = set(pp_df["id"])

common_ids = raw_ids.intersection(pp_ids)

print("RAW recipes:", len(raw_ids))
print("PP recipes:", len(pp_ids))

print("\nCommon recipe IDs:", len(common_ids))
print("RAW-only IDs:", len(raw_ids - pp_ids))
print("PP-only IDs:", len(pp_ids - raw_ids))

print(
    "\nPP coverage of RAW:",
    round(len(common_ids) / len(raw_ids) * 100, 2),
    "%"
)

RAW recipes: 231637
PP recipes: 178265

Common recipe IDs: 178265
RAW-only IDs: 53372
PP-only IDs: 0

PP coverage of RAW: 76.96 %


In [8]:
# --------------------------------------------------
# Step 6.3 - Inspect ingredient mapping
# --------------------------------------------------

import pickle

INGR_MAP_PATH = FOODCOM_DIR / "ingr_map.pkl"

with open(INGR_MAP_PATH, "rb") as file:
    ingr_map = pickle.load(file)

print("Mapping type:", type(ingr_map))
print("Mapping size:", len(ingr_map))

print("\nFirst 10 entries:")

if isinstance(ingr_map, dict):
    for i, (key, value) in enumerate(ingr_map.items()):
        if i == 10:
            break
        print(repr(key), "->", repr(value))

elif isinstance(ingr_map, list):
    print(ingr_map[:10])

ModuleNotFoundError: No module named 'pandas.core.indexes.numeric'

In [9]:
# --------------------------------------------------
# Step 7 - Inspect RAW recipe field structures
# --------------------------------------------------

import ast

columns_to_inspect = [
    "ingredients",
    "steps",
    "tags"
]

for column in columns_to_inspect:
    print("=" * 70)
    print("COLUMN:", column)

    sample = recipes_df[column].iloc[0]

    print("Python type:", type(sample))
    print("Raw value:", str(sample)[:500])

    try:
        parsed = ast.literal_eval(sample)
        print("Parsed type:", type(parsed))
        print("Parsed item count:", len(parsed))
        print("First 5 items:", parsed[:5])
    except (ValueError, SyntaxError, TypeError) as e:
        print("Parsing error:", e)

    print()

COLUMN: ingredients
Python type: <class 'str'>
Raw value: ['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt']
Parsed type: <class 'list'>
Parsed item count: 7
First 5 items: ['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter']

COLUMN: steps
Python type: <class 'str'>
Raw value: ['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fourths', 'remove seeds', 'for spicy squash , drizzle olive oil or melted butter over each cut squash piece', 'season with mexican seasoning mix ii', 'for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece', 'season with sweet mexican spice mix', 'bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin', 
Parsed type: <class 'list'>
Parsed item count: 11
First 5 items: ['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fo

In [10]:
# --------------------------------------------------
# Step 7.1 - Inspect Food.com nutrition values
# --------------------------------------------------

print("Nutrition example:")
print(recipes_df["nutrition"].iloc[0])

print("\nNutrition type:")
print(type(recipes_df["nutrition"].iloc[0]))

print("\nNutrition examples:")
for value in recipes_df["nutrition"].head(5):
    print(value)

Nutrition example:
[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]

Nutrition type:
<class 'str'>

Nutrition examples:
[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]
[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]
[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]
[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]
[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]


In [11]:
# --------------------------------------------------
# Step 8 - Parse Food.com list columns
# --------------------------------------------------

import ast
import pandas as pd

def parse_list(value):
    """
    Convert string representations of Python lists
    into actual Python lists.
    """
    if pd.isna(value):
        return []

    try:
        result = ast.literal_eval(value)

        if isinstance(result, list):
            return result

        return []

    except (ValueError, SyntaxError, TypeError):
        return []


foodcom_df = recipes_df.copy()

for column in ["ingredients", "steps", "tags", "nutrition"]:
    foodcom_df[column] = foodcom_df[column].apply(parse_list)

print("Original shape:", recipes_df.shape)
print("Structured shape:", foodcom_df.shape)

print("\nParsed column types:")

for column in ["ingredients", "steps", "tags", "nutrition"]:
    print(column, ":", type(foodcom_df[column].iloc[0]))

print("\nExample structured recipe:")

example = foodcom_df.iloc[0]

print("Name:", example["name"])
print("Ingredients:", example["ingredients"])
print("Steps:", example["steps"])
print("Tags:", example["tags"])
print("Nutrition:", example["nutrition"])

Original shape: (231637, 12)
Structured shape: (231637, 12)

Parsed column types:
ingredients : <class 'list'>
steps : <class 'list'>
tags : <class 'list'>
nutrition : <class 'list'>

Example structured recipe:
Name: arriba   baked winter squash mexican style
Ingredients: ['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt']
Steps: ['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fourths', 'remove seeds', 'for spicy squash , drizzle olive oil or melted butter over each cut squash piece', 'season with mexican seasoning mix ii', 'for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece', 'season with sweet mexican spice mix', 'bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin', 'be careful not to burn the squash especially if you opt to use sugar or butter', 'if you feel more comfortable , cover the sq

In [12]:
# --------------------------------------------------
# Step 8.1 - Validate parsed fields
# --------------------------------------------------

for column in ["ingredients", "steps", "tags", "nutrition"]:

    empty_count = foodcom_df[column].apply(
        lambda x: len(x) == 0
    ).sum()

    print(
        f"{column}: "
        f"{empty_count} empty records "
        f"({empty_count / len(foodcom_df) * 100:.2f}%)"
    )

# Check nutrition vector lengths
nutrition_lengths = foodcom_df["nutrition"].apply(len)

print("\nNutrition vector lengths:")
display(nutrition_lengths.value_counts().sort_index())

# Check valid nutrition vectors
valid_nutrition = foodcom_df["nutrition"].apply(
    lambda x: (
        len(x) == 7
        and all(
            isinstance(v, (int, float)) and pd.notna(v)
            for v in x
        )
    )
)

print("\nValid nutrition vectors:", valid_nutrition.sum())
print("Invalid nutrition vectors:", (~valid_nutrition).sum())

ingredients: 0 empty records (0.00%)
steps: 1 empty records (0.00%)
tags: 0 empty records (0.00%)
nutrition: 0 empty records (0.00%)

Nutrition vector lengths:


nutrition
7    231637
Name: count, dtype: int64


Valid nutrition vectors: 231637
Invalid nutrition vectors: 0


In [13]:
# --------------------------------------------------
# Step 9.1 - Build Food.com master recipe dataset
# --------------------------------------------------

master_df = foodcom_df.copy()

# Rename columns
master_df = master_df.rename(columns={
    "id": "recipe_id",
    "name": "recipe_name",
    "minutes": "preparation_time_min",
    "n_steps": "step_count",
    "n_ingredients": "ingredient_count"
})

# Extract calories (first value of nutrition vector)
master_df["calories_kcal"] = master_df["nutrition"].apply(
    lambda x: float(x[0]) if len(x) == 7 else None
)

# Quality indicators
master_df["has_description"] = (
    master_df["description"].fillna("").str.strip().ne("")
)

master_df["has_steps"] = master_df["steps"].apply(
    lambda x: len(x) > 0
)

master_df["has_ingredients"] = master_df["ingredients"].apply(
    lambda x: len(x) > 0
)

# Combined text representation for future NLP
def build_recipe_text(row):
    parts = [
        str(row["recipe_name"]),
        str(row["description"]) if pd.notna(row["description"]) else "",
        "Ingredients: " + ", ".join(row["ingredients"]),
        "Preparation: " + " ".join(row["steps"])
    ]

    return " ".join(part for part in parts if part.strip())

master_df["recipe_text"] = master_df.apply(
    build_recipe_text,
    axis=1
)

print("Master dataset shape:", master_df.shape)

print("\nNew columns:")
print(master_df.columns.tolist())

print("\nExample master recipe:")
display(
    master_df[[
        "recipe_id",
        "recipe_name",
        "description",
        "ingredients",
        "calories_kcal",
        "has_description",
        "has_steps",
        "recipe_text"
    ]].head(1)
)

Master dataset shape: (231637, 17)

New columns:
['recipe_name', 'recipe_id', 'preparation_time_min', 'contributor_id', 'submitted', 'tags', 'nutrition', 'step_count', 'steps', 'description', 'ingredients', 'ingredient_count', 'calories_kcal', 'has_description', 'has_steps', 'has_ingredients', 'recipe_text']

Example master recipe:


,recipe_id,recipe_name,description,ingredients,calories_kcal,has_description,has_steps,recipe_text
0,137739,arriba baked winter squash mexican style,autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",51.5,True,True,arriba baked winter squash mexican style aut...


In [14]:
# --------------------------------------------------
# Step 9.2 - Validate master dataset
# --------------------------------------------------

print("Total recipes:", len(master_df))

print("\nMissing recipe IDs:", master_df["recipe_id"].isna().sum())
print("Duplicate recipe IDs:", master_df["recipe_id"].duplicated().sum())

print("\nQuality indicators:")
print(master_df[
    ["has_description", "has_ingredients", "has_steps"]
].sum())

print("\nMissing calories:", master_df["calories_kcal"].isna().sum())

print("\nRecipe text statistics:")
print(master_df["recipe_text"].str.split().str.len().describe())

Total recipes: 231637

Missing recipe IDs: 0
Duplicate recipe IDs: 0

Quality indicators:
has_description    226655
has_ingredients    231637
has_steps          231636
dtype: int64

Missing calories: 0

Recipe text statistics:
count    231637.000000
mean        160.548518
std          91.116369
min          11.000000
25%         100.000000
50%         141.000000
75%         198.000000
max        2344.000000
Name: recipe_text, dtype: float64


In [15]:
# --------------------------------------------------
# Step 9.3 - Improve combined recipe text
# --------------------------------------------------

def build_recipe_text(row):

    description = (
        str(row["description"])
        if pd.notna(row["description"])
        else ""
    )

    ingredients = ", ".join(row["ingredients"])
    steps = " ".join(row["steps"])

    return (
        f"Recipe: {row['recipe_name']}. "
        f"Description: {description}. "
        f"Ingredients: {ingredients}. "
        f"Preparation: {steps}"
    )


master_df["recipe_text"] = master_df.apply(
    build_recipe_text,
    axis=1
)

print(master_df.loc[0, "recipe_text"])

Recipe: arriba   baked winter squash mexican style. Description: autumn is my favorite time of year to cook! this recipe 
can be prepared either spicy or sweet, your choice!
two of my posted mexican-inspired seasoning mix recipes are offered as suggestions.. Ingredients: winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt. Preparation: make a choice and proceed with recipe depending on size of squash , cut into half or fourths remove seeds for spicy squash , drizzle olive oil or melted butter over each cut squash piece season with mexican seasoning mix ii for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece season with sweet mexican spice mix bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin be careful not to burn the squash especially if you opt to use sugar or butter if you feel more comfortable , cover the squash with aluminum foil the first half 

In [16]:
# --------------------------------------------------
# Step 9.4 - Save Food.com master dataset
# --------------------------------------------------

PROCESSED_RECIPES_DIR = (
    PROJECT_ROOT / "data" / "processed" / "recipes"
)

PROCESSED_RECIPES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    PROCESSED_RECIPES_DIR / "foodcom_recipes_master.csv"
)

master_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved successfully:", OUTPUT_PATH)
print("File size:", round(OUTPUT_PATH.stat().st_size / (1024**2), 2), "MB")

Saved successfully: C:\Users\dell\NutriVision\data\processed\recipes\foodcom_recipes_master.csv
File size: 491.3 MB


In [19]:
# --------------------------------------------------
# Step 10.1 - Description length distribution
# --------------------------------------------------

bins = [0, 5, 10, 20, 50, 100, 200, float("inf")]

labels = [
    "1-5 words",
    "6-10 words",
    "11-20 words",
    "21-50 words",
    "51-100 words",
    "101-200 words",
    "More than 200 words"
]

description_categories = pd.cut(
    description_lengths,
    bins=bins,
    labels=labels,
    include_lowest=True
)

display(
    description_categories.value_counts()
    .sort_index()
)

print("\nEmpty descriptions:", (description_lengths == 0).sum())

description
1-5 words              18763
6-10 words             22264
11-20 words            45794
21-50 words            93355
51-100 words           41738
101-200 words           9007
More than 200 words      716
Name: count, dtype: int64


Empty descriptions: 4982


In [20]:
# Correct description length distribution

bins = [-0.1, 0, 5, 10, 20, 50, 100, 200, float("inf")]

labels = [
    "Empty",
    "1-5 words",
    "6-10 words",
    "11-20 words",
    "21-50 words",
    "51-100 words",
    "101-200 words",
    "More than 200 words"
]

description_categories = pd.cut(
    description_lengths,
    bins=bins,
    labels=labels
)

display(
    description_categories.value_counts()
    .reindex(labels)
)

description
Empty                   4982
1-5 words              13781
6-10 words             22264
11-20 words            45794
21-50 words            93355
51-100 words           41738
101-200 words           9007
More than 200 words      716
Name: count, dtype: int64

In [21]:
# Step 10.2 - Inspect descriptions by length

for min_words, max_words, label in [
    (1, 5, "Very short"),
    (6, 15, "Short"),
    (16, 50, "Medium"),
    (51, 100, "Long"),
]:
    mask = (
        (description_lengths >= min_words)
        & (description_lengths <= max_words)
    )

    sample = master_df[mask].sample(
        min(5, mask.sum()),
        random_state=42
    )

    print("\n" + "=" * 80)
    print(label, "descriptions")

    for _, row in sample.iterrows():
        print("\nRecipe:", row["recipe_name"])
        print("Description:", row["description"])


Very short descriptions

Recipe: simple pancake recipe
Description: just like it says.  simple...

Recipe: mixed berry cornmeal cobbler
Description: extremely fruity, holiday treat

Recipe: ice cream pancake with honey and walnuts
Description: from the 'oriental menu planner'

Recipe: almond  crusted chicken breasts
Description: yum!

Recipe: greek revolution shooter
Description: another yummy ouzo shot!

Short descriptions

Recipe: healthy maple walnut baked oatmeal
Description: s easy and great to have in the fridge for a quick weekday breakfast.

Recipe: crock pot minestrone soup
Description: i found this on another site-sounds tasty & healthy!

Recipe: thai pumpkin and coconut cream soup
Description: a little different from the ones already posted.

Recipe: polpette and orzo in broth
Description: this is a fast and easy italian soup. from mark bittman.

Recipe: shrimp and feta greek style pizza
Description: makes a very gourmet type pizza. you can't get this one at pizza hut!

Med

In [22]:
# Step 10.3 - Build two recipe text representations

def build_recipe_text_core(row):
    ingredients = ", ".join(row["ingredients"])
    steps = " ".join(row["steps"])

    return (
        f"Recipe: {row['recipe_name']}. "
        f"Ingredients: {ingredients}. "
        f"Preparation: {steps}"
    )


def build_recipe_text_with_description(row):
    description = (
        str(row["description"])
        if pd.notna(row["description"])
        else ""
    )

    return (
        f"Recipe: {row['recipe_name']}. "
        f"Description: {description}. "
        f"Ingredients: {', '.join(row['ingredients'])}. "
        f"Preparation: {' '.join(row['steps'])}"
    )


master_df["recipe_text_core"] = master_df.apply(
    build_recipe_text_core,
    axis=1
)

master_df["recipe_text_with_description"] = master_df.apply(
    build_recipe_text_with_description,
    axis=1
)

print("Dataset shape:", master_df.shape)

print("\nCORE TEXT EXAMPLE:")
print(master_df.iloc[0]["recipe_text_core"])

print("\nTEXT WITH DESCRIPTION EXAMPLE:")
print(master_df.iloc[0]["recipe_text_with_description"])

Dataset shape: (231637, 19)

CORE TEXT EXAMPLE:
Recipe: arriba   baked winter squash mexican style. Ingredients: winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt. Preparation: make a choice and proceed with recipe depending on size of squash , cut into half or fourths remove seeds for spicy squash , drizzle olive oil or melted butter over each cut squash piece season with mexican seasoning mix ii for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece season with sweet mexican spice mix bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin be careful not to burn the squash especially if you opt to use sugar or butter if you feel more comfortable , cover the squash with aluminum foil the first half hour , give or take , of baking if desired , season with salt

TEXT WITH DESCRIPTION EXAMPLE:
Recipe: arriba   baked winter squash mexican style. Description: 

In [23]:
# Step 10.4 - Analyze core text completeness

core_word_counts = (
    master_df["recipe_text_core"]
    .fillna("")
    .str.split()
    .str.len()
)

description_word_counts = (
    master_df["recipe_text_with_description"]
    .fillna("")
    .str.split()
    .str.len()
)

print("CORE TEXT STATISTICS")
display(core_word_counts.describe())

print("\nTEXT WITH DESCRIPTION STATISTICS")
display(description_word_counts.describe())

print("\nEmpty core texts:", (core_word_counts == 0).sum())
print("Core texts below 20 words:", (core_word_counts < 20).sum())

print(
    "\nCore text missing values:",
    master_df["recipe_text_core"].isna().sum()
)

print(
    "Text with description missing values:",
    master_df["recipe_text_with_description"].isna().sum()
)

CORE TEXT STATISTICS


count    231637.000000
mean        125.944918
std          77.569842
min           8.000000
25%          75.000000
50%         109.000000
75%         157.000000
max        2279.000000
Name: recipe_text_core, dtype: float64


TEXT WITH DESCRIPTION STATISTICS


count    231637.000000
mean        162.582623
std          91.103808
min          13.000000
25%         102.000000
50%         143.000000
75%         200.000000
max        2346.000000
Name: recipe_text_with_description, dtype: float64


Empty core texts: 0
Core texts below 20 words: 511

Core text missing values: 0
Text with description missing values: 0


In [24]:
# Step 10.5 - Inspect recipes with fewer than 20 words

short_recipes = master_df.loc[
    core_word_counts < 20,
    [
        "recipe_id",
        "recipe_name",
        "ingredients",
        "steps",
        "recipe_text_core"
    ]
]

print("Short recipes:", len(short_recipes))

display(short_recipes.head(20))

Short recipes: 511


,recipe_id,recipe_name,ingredients,steps,recipe_text_core
328,28951,berry good smoothie,"[apple juice, strawberry, raspberries, blackbe...",[mix all ingredients in blender until smooth],Recipe: berry good smoothie. Ingredients: app...
617,238453,old style pancakes,"[bisquick, eggs, beer]",[beat mixture and fry on griddle],Recipe: old style pancakes. Ingredients: bisq...
768,33246,1 2 3 bread,"[beer, sugar, self rising flour]",[bake 50 minutes at 400 degrees],"Recipe: 1 2 3 bread. Ingredients: beer, sugar,..."
995,262448,17 twist,"[raspberry vodka, mountain dew soda]",[mix lightly and enjoy],Recipe: 17 twist. Ingredients: raspberry vodka...
1385,474672,30 second steak sauce,"[ketchup, worcestershire sauce, onion powder, ...","[mix all ingredients together, serve]",Recipe: 30 second steak sauce. Ingredients: ke...
2281,225076,absolut trouble,"[absolut citron vodka, grand marnier, grenadin...","[shake ingredients with ice, serve]",Recipe: absolut trouble. Ingredients: absolut ...
2400,219479,acapulco gold virgin cocktail,"[pineapple juice, coconut cream, whipping crea...","[blend all ingredients, serve]",Recipe: acapulco gold virgin cocktail. Ingre...
2615,157582,adult beverage,"[kahlua, baileys irish cream]","[layer in order, shoot !]","Recipe: adult beverage. Ingredients: kahlua, b..."
2839,293213,afterglow,"[grenadine, orange juice, pineapple juice]","[mix, serve over ice]","Recipe: afterglow. Ingredients: grenadine, ora..."
2847,485158,aftershock,"[drambuie, malibu rum, cherry brandy, lemonade]",[just pour the ingredients into a glass and stir],"Recipe: aftershock. Ingredients: drambuie, mal..."


In [25]:
# Step 10.6 - Analyze the most frequent recipe tags

from collections import Counter

tag_counter = Counter(
    tag
    for tags in master_df["tags"]
    for tag in tags
)

top_tags = pd.DataFrame(
    tag_counter.most_common(50),
    columns=["tag", "recipe_count"]
)

display(top_tags)

print("\nUnique tags:", len(tag_counter))

,tag,recipe_count
0,preparation,230546
1,time-to-make,225326
2,course,218148
3,main-ingredient,170446
4,dietary,165091
5,easy,126062
6,occasion,114145
7,cuisine,91165
8,low-in-something,85776
9,main-dish,71786



Unique tags: 552


In [26]:
# Step 10.7 - Inspect food-related tags

food_related_tags = [
    "course",
    "main-ingredient",
    "main-dish",
    "desserts",
    "side-dishes",
    "appetizers",
    "beverages",
    "breakfast",
    "lunch",
    "dinner",
    "meat",
    "poultry",
    "seafood",
    "vegetables",
    "fruit",
    "pasta-rice-and-grains",
    "eggs-dairy"
]

for target_tag in food_related_tags:
    matching_recipes = master_df[
        master_df["tags"].apply(
            lambda tags: target_tag in tags
        )
    ]

    print(f"\n{target_tag}: {len(matching_recipes):,}")

    if not matching_recipes.empty:
        print(
            matching_recipes["recipe_name"]
            .head(5)
            .tolist()
        )


course: 218,148
['arriba   baked winter squash mexican style', 'a bit different  breakfast pizza', 'all in the kitchen  chili', 'alouette  potatoes', 'amish  tomato ketchup  for canning']

main-ingredient: 170,446
['arriba   baked winter squash mexican style', 'a bit different  breakfast pizza', 'alouette  potatoes', 'amish  tomato ketchup  for canning', 'apple a day  milk shake']

main-dish: 71,786
['a bit different  breakfast pizza', 'all in the kitchen  chili', 'backyard style  barbecued ribs', 'chicken lickin  good  pork chops', 'chinese  chop suey']

desserts: 43,203
['bananas 4 ice cream  pie', 'better than sex  strawberries', 'chinese  candy', 'deep fried dessert thingys', 'get the sensation  brownies']

side-dishes: 26,902
['arriba   baked winter squash mexican style', 'alouette  potatoes', 'better then bush s  baked beans', 'chile rellenos', 'fried  potatoes']

appetizers: 20,379
['aww  marinated olives', 'chile rellenos', 'grilled  ranch bread', 'i can t believe it s spinach

In [27]:
# Step 10.8 - Inspect actual tag values

tag_groups = [
    "course",
    "main-ingredient"
]

for group in tag_groups:

    values = master_df["tags"].apply(
        lambda tags: [
            tag for tag in tags
            if tag.startswith(group + ":")
        ]
    )

    counter = Counter(
        tag
        for row in values
        for tag in row
    )

    print("\n" + "=" * 60)
    print(f"Most common values for {group}")

    display(
        pd.DataFrame(
            counter.most_common(30),
            columns=["tag_value", "recipe_count"]
        )
    )


Most common values for course


,tag_value,recipe_count



Most common values for main-ingredient


,tag_value,recipe_count


In [28]:
# Inspect the tags of the first 10 recipes

for _, row in master_df.head(10).iterrows():
    print("\nRecipe:", row["recipe_name"])
    print("Tags:", row["tags"])


Recipe: arriba   baked winter squash mexican style
Tags: ['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'side-dishes', 'vegetables', 'mexican', 'easy', 'fall', 'holiday-event', 'vegetarian', 'winter', 'dietary', 'christmas', 'seasonal', 'squash']

Recipe: a bit different  breakfast pizza
Tags: ['30-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'breakfast', 'main-dish', 'pork', 'american', 'oven', 'easy', 'kid-friendly', 'pizza', 'dietary', 'northeastern-united-states', 'meat', 'equipment']

Recipe: all in the kitchen  chili
Tags: ['time-to-make', 'course', 'preparation', 'main-dish', 'chili', 'crock-pot-slow-cooker', 'dietary', 'equipment', '4-hours-or-less']

Recipe: alouette  potatoes
Tags: ['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'preparation', 'occasion', 'side-dishes', 'eggs-dairy', 'potatoes', 'vegetables

In [29]:
# Find tags containing course-related keywords

course_keywords = [
    "main-course",
    "breakfast",
    "dessert",
    "appetizer",
    "beverage",
    "side-dish",
    "lunch",
    "dinner"
]

for keyword in course_keywords:
    count = sum(
        keyword in tag
        for tags in master_df["tags"]
        for tag in tags
    )
    print(f"{keyword}: {count:,}")

main-course: 0
breakfast: 13,660
dessert: 45,160
appetizer: 20,379
beverage: 11,124
side-dish: 26,906
lunch: 23,811
dinner: 37,561


In [30]:
# Step 10.9 - Inspect useful descriptive tags

generic_tags = {
    "course",
    "main-ingredient",
    "cuisine",
    "preparation",
    "occasion",
    "dietary",
    "equipment",
    "time-to-make",
    "number-of-servings",
    "taste-mood",
    "low-in-something",
    "technique",
    "presentation"
}

descriptive_tag_counter = Counter(
    tag
    for tags in master_df["tags"]
    for tag in tags
    if tag not in generic_tags
)

display(
    pd.DataFrame(
        descriptive_tag_counter.most_common(50),
        columns=["tag", "recipe_count"]
    )
)

,tag,recipe_count
0,easy,126062
1,main-dish,71786
2,60-minutes-or-less,69990
3,meat,56042
4,30-minutes-or-less,55077
5,vegetables,53814
6,4-hours-or-less,49497
7,north-american,48479
8,3-steps-or-less,44933
9,15-minutes-or-less,43934


In [31]:
# Step 10.10 - Analyze ingredients and preparation steps

ingredient_counts = master_df["ingredients"].apply(len)
step_counts = master_df["steps"].apply(len)

print("INGREDIENT COUNT STATISTICS")
display(ingredient_counts.describe())

print("\nPREPARATION STEP COUNT STATISTICS")
display(step_counts.describe())

print("\nRecipes with fewer than 2 ingredients:",
      (ingredient_counts < 2).sum())

print("Recipes with fewer than 2 preparation steps:",
      (step_counts < 2).sum())

print("Recipes with more than 30 ingredients:",
      (ingredient_counts > 30).sum())

print("Recipes with more than 30 preparation steps:",
      (step_counts > 30).sum())

INGREDIENT COUNT STATISTICS


count    231637.000000
mean          9.051153
std           3.734796
min           1.000000
25%           6.000000
50%           9.000000
75%          11.000000
max          43.000000
Name: ingredients, dtype: float64


PREPARATION STEP COUNT STATISTICS


count    231637.000000
mean          9.765499
std           5.995128
min           0.000000
25%           6.000000
50%           9.000000
75%          12.000000
max         145.000000
Name: steps, dtype: float64


Recipes with fewer than 2 ingredients: 25
Recipes with fewer than 2 preparation steps: 2443
Recipes with more than 30 ingredients: 37
Recipes with more than 30 preparation steps: 2165


In [32]:
# Step 10.11 - Inspect unusual recipes

conditions = {
    "No preparation steps": step_counts == 0,
    "One preparation step": step_counts == 1,
    "More than 30 ingredients": ingredient_counts > 30,
    "More than 30 preparation steps": step_counts > 30
}

for label, mask in conditions.items():

    print("\n" + "=" * 80)
    print(label)
    print("Number of recipes:", mask.sum())

    sample = master_df.loc[
        mask,
        [
            "recipe_id",
            "recipe_name",
            "ingredients",
            "steps"
        ]
    ].head(5)

    display(sample)


No preparation steps
Number of recipes: 1


,recipe_id,recipe_name,ingredients,steps
3381,176767,all season bread,"[flour, baking soda, salt, baking powder, cinn...",[]



One preparation step
Number of recipes: 2442


,recipe_id,recipe_name,ingredients,steps
92,83133,stove top bbq beef or pork ribs,"[beef ribs, onion, ketchup, water, fresh lemon...",[cut all ribs into serving size pieces sprinkl...
249,343448,outback steak rub,"[salt, paprika, ground black pepper, onion pow...","[mix , sprinkle lightly on your favorite cut o..."
328,28951,berry good smoothie,"[apple juice, strawberry, raspberries, blackbe...",[mix all ingredients in blender until smooth]
520,181808,sexy sauce,"[dry red wine, ketchup, dark molasses, dijon m...","[whisk all ingredients together , bring to boi..."
531,358626,souped up veggie dip,"[knorr vegetable soup mix, sour cream, tabasco...",[mix together and dip dip dip !]



More than 30 ingredients
Number of recipes: 37


,recipe_id,recipe_name,ingredients,steps
2162,3748,a to z bread,"[sugar, vegetable oil, eggs, vanilla extract, ...","[in a mixing bowl , combine the sugar , oil an..."
2532,117155,add ins bread,"[flour, cinnamon, eggs, sugar, vanilla, salt, ...","[sift dry ingredients, set aside, beat eggs in..."
5231,110887,andalusian condiment soup,"[butter, olive oil, onions, flour, tomato pure...","[melt butter in a large stock pot, add oil and..."
10808,389770,authentic mexican tamale pie,"[ground beef, lean pork sausage, vegetable oil...","[prepare chili:, in a large pot brown ground b..."
18352,248707,basic fried rice with variations,"[brown rice, green onions, oil, soy sauce, sal...","[dice or shred meat and vegetables, heat oil i..."



More than 30 preparation steps
Number of recipes: 2165


,recipe_id,recipe_name,ingredients,steps
69,38276,now and later vegetarian empanadas,"[carrots, butter, onion, sliced mushrooms, zuc...","[in a pot , cover carrots with water to cover,..."
162,310570,burek or feta cheese phyllo pie,"[phyllo dough, eggs, milk, feta, oil, cottage ...","[first - let me explain the feta, we have an i..."
207,293225,honey i m peanuts about you cake,"[unsweetened cocoa, espresso, buttermilk, unbl...","[cook's, preheat the oven to 350 degrees f, gr..."
283,129581,tastes like mounds bar kahlua cake,"[devil's food cake mix, instant chocolate pudd...","[preheat over to 350f, put all ingredients for..."
316,242,any muffins,"[white flour, milk, fat, egg, sugar, baking po...","[bake 15 to 20 minutes in a 400 oven, for the ..."


In [33]:
# Step 10.12 - Add recipe quality indicators

master_df["has_minimum_ingredients"] = (
    master_df["ingredients"].apply(len) >= 2
)

master_df["has_preparation_steps"] = (
    master_df["steps"].apply(len) > 0
)

master_df["has_multiple_steps"] = (
    master_df["steps"].apply(len) >= 2
)

master_df["has_valid_core_text"] = (
    master_df["recipe_text_core"]
    .fillna("")
    .str.strip()
    .ne("")
)

print("QUALITY INDICATORS")
display(
    master_df[
        [
            "has_minimum_ingredients",
            "has_preparation_steps",
            "has_multiple_steps",
            "has_valid_core_text"
        ]
    ].sum()
)

print("\nTotal recipes:", len(master_df))

QUALITY INDICATORS


has_minimum_ingredients    231612
has_preparation_steps      231636
has_multiple_steps         229194
has_valid_core_text        231637
dtype: int64


Total recipes: 231637


In [34]:
# Step 10.13 - Save updated Food.com master dataset

from pathlib import Path

OUTPUT_PATH = Path(
    r"C:\Users\dell\NutriVision\data\processed\recipes\foodcom_recipes_master.csv"
)

master_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved successfully:", OUTPUT_PATH)
print("Dataset shape:", master_df.shape)
print("File size (MB):", round(OUTPUT_PATH.stat().st_size / (1024 ** 2), 2))

Saved successfully: C:\Users\dell\NutriVision\data\processed\recipes\foodcom_recipes_master.csv
Dataset shape: (231637, 23)
File size (MB): 860.72


In [35]:
# --------------------------------------------------
# Step 10.14 - Reload and validate the master dataset
# --------------------------------------------------

import pandas as pd
import ast

reloaded_df = pd.read_csv(OUTPUT_PATH)

print("Dataset shape:", reloaded_df.shape)

print("\nColumns:")
print(reloaded_df.columns.tolist())

print("\nMissing values:")
display(
    reloaded_df.isna().sum().sort_values(ascending=False)
)

print("\nFirst recipe:")
display(reloaded_df.iloc[0])

Dataset shape: (231637, 23)

Columns:
['recipe_name', 'recipe_id', 'preparation_time_min', 'contributor_id', 'submitted', 'tags', 'nutrition', 'step_count', 'steps', 'description', 'ingredients', 'ingredient_count', 'calories_kcal', 'has_description', 'has_steps', 'has_ingredients', 'recipe_text', 'recipe_text_core', 'recipe_text_with_description', 'has_minimum_ingredients', 'has_preparation_steps', 'has_multiple_steps', 'has_valid_core_text']

Missing values:


description                     4979
recipe_name                        1
recipe_id                          0
contributor_id                     0
submitted                          0
tags                               0
preparation_time_min               0
nutrition                          0
step_count                         0
steps                              0
ingredients                        0
ingredient_count                   0
calories_kcal                      0
has_description                    0
has_steps                          0
has_ingredients                    0
recipe_text                        0
recipe_text_core                   0
recipe_text_with_description       0
has_minimum_ingredients            0
has_preparation_steps              0
has_multiple_steps                 0
has_valid_core_text                0
dtype: int64


First recipe:


recipe_name                            arriba   baked winter squash mexican style
recipe_id                                                                  137739
preparation_time_min                                                           55
contributor_id                                                              47892
submitted                                                              2005-09-16
tags                            ['60-minutes-or-less', 'time-to-make', 'course...
nutrition                                   [51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]
step_count                                                                     11
steps                           ['make a choice and proceed with recipe', 'dep...
description                     autumn is my favorite time of year to cook! th...
ingredients                     ['winter squash', 'mexican seasoning', 'mixed ...
ingredient_count                                                                7
calories_kcal   

In [36]:
# --------------------------------------------------
# Step 10.15 - Inspect recipes with missing names
# --------------------------------------------------

missing_name_df = reloaded_df[
    reloaded_df["recipe_name"].isna()
]

print("Recipes with missing names:", len(missing_name_df))

display(
    missing_name_df[
        [
            "recipe_id",
            "preparation_time_min",
            "ingredients",
            "steps",
            "description",
            "recipe_text_core"
        ]
    ]
)

Recipes with missing names: 1


,recipe_id,preparation_time_min,ingredients,steps,description,recipe_text_core
721,368257,10,"['lemon', 'honey', 'horseradish mustard', 'gar...","['in a bowl , combine ingredients except for o...",-------------,"Recipe: nan. Ingredients: lemon, honey, horser..."


In [37]:
# --------------------------------------------------
# Step 10.16 - Handle missing recipe names
# --------------------------------------------------

missing_name_mask = (
    reloaded_df["recipe_name"].isna()
    | reloaded_df["recipe_name"].fillna("").str.strip().eq("")
)

reloaded_df.loc[
    missing_name_mask,
    "recipe_name"
] = (
    "Unnamed Recipe "
    + reloaded_df.loc[missing_name_mask, "recipe_id"].astype(str)
)

print("Remaining missing names:", reloaded_df["recipe_name"].isna().sum())

display(
    reloaded_df.loc[
        missing_name_mask,
        ["recipe_id", "recipe_name", "ingredients", "steps"]
    ]
)

Remaining missing names: 0


,recipe_id,recipe_name,ingredients,steps
721,368257,Unnamed Recipe 368257,"['lemon', 'honey', 'horseradish mustard', 'gar...","['in a bowl , combine ingredients except for o..."


In [38]:
# Rebuild text representations after correcting the recipe name

def build_recipe_text_core(row):
    ingredients = ", ".join(row["ingredients"])
    steps = " ".join(row["steps"])

    return (
        f"Recipe: {row['recipe_name']}. "
        f"Ingredients: {ingredients}. "
        f"Preparation: {steps}"
    )


def build_recipe_text_with_description(row):
    description = (
        str(row["description"])
        if pd.notna(row["description"])
        else ""
    )

    ingredients = ", ".join(row["ingredients"])
    steps = " ".join(row["steps"])

    return (
        f"Recipe: {row['recipe_name']}. "
        f"Description: {description}. "
        f"Ingredients: {ingredients}. "
        f"Preparation: {steps}"
    )


reloaded_df["recipe_text_core"] = reloaded_df.apply(
    build_recipe_text_core,
    axis=1
)

reloaded_df["recipe_text_with_description"] = reloaded_df.apply(
    build_recipe_text_with_description,
    axis=1
)

reloaded_df["recipe_text"] = reloaded_df["recipe_text_core"]

print("Missing names:", reloaded_df["recipe_name"].isna().sum())
print("Missing core text:", reloaded_df["recipe_text_core"].isna().sum())

Missing names: 0
Missing core text: 0


In [39]:
reloaded_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Updated dataset saved:", OUTPUT_PATH)
print("Final shape:", reloaded_df.shape)
print("File size (MB):", round(OUTPUT_PATH.stat().st_size / (1024 ** 2), 2))

Updated dataset saved: C:\Users\dell\NutriVision\data\processed\recipes\foodcom_recipes_master.csv
Final shape: (231637, 23)
File size (MB): 1403.01
